# 02 — RAG Query

This notebook implements the end-to-end RAG query pipeline. A natural language question is embedded, matched against the Milvus vector store, and a grounded answer is generated by the Granite 3.1 8B Instruct model served via vLLM.

**Run notebook 01-ingest-and-embed.ipynb before this one.**

## Prerequisites
- Notebook 01 completed successfully — Milvus collection populated
- Granite InferenceService in `Ready` state
- All environment variables set in the workbench

## Environment variables required

| Variable | Description |
|---|---|
| `MINIO_ACCESS_KEY` | MinIO access key |
| `MINIO_SECRET_KEY` | MinIO secret key |
| `MILVUS_HOST` | Milvus service host |
| `MILVUS_PORT` | Milvus gRPC port — default `19530` |
| `INFERENCE_ENDPOINT` | vLLM InferenceService URL — retrieve with `oc get inferenceservice granite-instruct -n sovereign-rag -o jsonpath='{.status.url}'` |
| `INFERENCE_TOKEN` | Bearer token for the InferenceService — retrieve from RHOAI dashboard or via `oc` |

## Cell 1 — Install dependencies

In [ ]:
import sys

!{sys.executable} -m pip install --quiet \
    pymilvus==2.4.3 \
    sentence-transformers==3.0.1 \
    openai==1.35.0

print("Dependencies installed.")

## Cell 2 — Configuration

In [ ]:
import os

# Milvus
MILVUS_HOST       = os.environ.get("MILVUS_HOST", "milvus.sovereign-rag.svc.cluster.local")
MILVUS_PORT       = int(os.environ.get("MILVUS_PORT", "19530"))
MILVUS_COLLECTION = "regulatory_docs"

# Embedding — must match notebook 01
EMBEDDING_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM     = 384

# Retrieval
TOP_K             = 5   # number of chunks to retrieve
NPROBE            = 16  # IVF search parameter — higher = more accurate, slower

# Inference — vLLM exposes an OpenAI-compatible API
INFERENCE_ENDPOINT = os.environ["INFERENCE_ENDPOINT"]  # fail fast if not set
INFERENCE_TOKEN    = os.environ.get("INFERENCE_TOKEN", "")  # may be empty if auth disabled
MODEL_NAME         = "granite-3.1-8b-instruct"  # must match the model loaded by vLLM

# Generation parameters
MAX_TOKENS        = 1024
TEMPERATURE       = 0.1   # low temperature for factual regulatory Q&A

print("Configuration loaded.")
print(f"  Milvus         : {MILVUS_HOST}:{MILVUS_PORT}")
print(f"  Collection     : {MILVUS_COLLECTION}")
print(f"  Top-K          : {TOP_K}")
print(f"  Inference      : {INFERENCE_ENDPOINT}")
print(f"  Model          : {MODEL_NAME}")
print(f"  Temperature    : {TEMPERATURE}")

## Cell 3 — Connect to Milvus and load embedding model

In [ ]:
from pymilvus import connections, Collection, utility
from sentence_transformers import SentenceTransformer

# Connect to Milvus
connections.connect(
    alias="default",
    host=MILVUS_HOST,
    port=MILVUS_PORT
)

# Verify collection exists
if not utility.has_collection(MILVUS_COLLECTION):
    raise RuntimeError(
        f"Collection '{MILVUS_COLLECTION}' not found in Milvus. "
        f"Run notebook 01-ingest-and-embed.ipynb first."
    )

collection = Collection(name=MILVUS_COLLECTION)
collection.load()

print(f"Connected to Milvus. Collection '{MILVUS_COLLECTION}' loaded.")
print(f"  Vector count: {collection.num_entities}")

# Load embedding model
print(f"\nLoading embedding model: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model loaded.")

## Cell 4 — Connect to Granite via vLLM

vLLM exposes an OpenAI-compatible API. The `openai` client is pointed at the internal InferenceService URL — no external calls.

In [ ]:
from openai import OpenAI

# Initialise OpenAI-compatible client pointed at the vLLM endpoint
llm_client = OpenAI(
    base_url=f"{INFERENCE_ENDPOINT}/v1",
    api_key=INFERENCE_TOKEN if INFERENCE_TOKEN else "ignored"  # vLLM requires a value
)

# Verify the model is available
available_models = [m.id for m in llm_client.models.list().data]
print(f"Models available at endpoint: {available_models}")

if MODEL_NAME not in available_models:
    print(f"\nWARNING: '{MODEL_NAME}' not in available models list.")
    print(f"Available: {available_models}")
    print("Update MODEL_NAME in Cell 2 to match the loaded model.")
else:
    print(f"\nModel '{MODEL_NAME}' confirmed available. Ready to query.")

## Cell 5 — Define the RAG pipeline

Three functions:
- `retrieve()` — embeds the query and fetches top-K chunks from Milvus
- `build_prompt()` — constructs a grounded prompt from retrieved context
- `query()` — orchestrates the full pipeline and returns the answer with sources

In [ ]:
from dataclasses import dataclass


@dataclass
class RetrievedChunk:
    text: str
    source: str
    page: int
    score: float


@dataclass
class RAGResponse:
    question: str
    answer: str
    sources: list[RetrievedChunk]
    model: str


def retrieve(question: str, top_k: int = TOP_K) -> list[RetrievedChunk]:
    """
    Embeds the question and retrieves the top-K most relevant chunks from Milvus.
    """
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).tolist()

    results = collection.search(
        data=query_embedding,
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": NPROBE}},
        limit=top_k,
        output_fields=["text", "source", "page"]
    )

    chunks = []
    for hit in results[0]:
        chunks.append(RetrievedChunk(
            text=hit.entity.get("text"),
            source=hit.entity.get("source"),
            page=hit.entity.get("page"),
            score=hit.score
        ))
    return chunks


def build_prompt(question: str, chunks: list[RetrievedChunk]) -> str:
    """
    Constructs a grounded prompt from the retrieved context chunks.
    Instructs the model to answer only from provided context and
    to cite sources — important for a regulated-environment use case.
    """
    context_blocks = []
    for i, chunk in enumerate(chunks, start=1):
        context_blocks.append(
            f"[Source {i}: {chunk.source}, page {chunk.page}]\n{chunk.text}"
        )
    context = "\n\n".join(context_blocks)

    prompt = f"""You are a regulatory compliance assistant for a financial institution.
Answer the question using only the context provided below.
If the context does not contain enough information to answer the question, say so clearly.
Always cite the source document and page number for each claim you make.
Do not speculate or draw on knowledge outside the provided context.

Context:
{context}

Question: {question}

Answer:"""
    return prompt


def query(question: str, top_k: int = TOP_K) -> RAGResponse:
    """
    Full RAG pipeline: retrieve -> prompt -> generate -> return.
    """
    # Step 1: Retrieve relevant chunks
    chunks = retrieve(question, top_k=top_k)

    # Step 2: Build grounded prompt
    prompt = build_prompt(question, chunks)

    # Step 3: Generate answer via Granite
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a precise regulatory compliance assistant. "
                           "Answer only from the provided context. Cite sources."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE
    )

    answer = response.choices[0].message.content.strip()
    model_used = response.model

    return RAGResponse(
        question=question,
        answer=answer,
        sources=chunks,
        model=model_used
    )


print("RAG pipeline functions defined.")
print("  retrieve()     — embed query + Milvus search")
print("  build_prompt() — construct grounded prompt with source attribution")
print("  query()        — full pipeline orchestration")

## Cell 6 — Helper: display results

A display function that formats results clearly — useful for demos.

In [ ]:
def display_response(response: RAGResponse, show_chunks: bool = True) -> None:
    """
    Formats and prints a RAGResponse clearly.
    Set show_chunks=False to suppress retrieved chunk detail for a cleaner demo view.
    """
    print("=" * 70)
    print(f"QUESTION")
    print(f"  {response.question}")
    print()
    print(f"ANSWER  (model: {response.model})")
    print("-" * 70)
    print(response.answer)
    print()

    if show_chunks:
        print("-" * 70)
        print(f"RETRIEVED CONTEXT  ({len(response.sources)} chunks)")
        for i, chunk in enumerate(response.sources, start=1):
            print(f"\n  [{i}] {chunk.source}, page {chunk.page}  "
                  f"(similarity: {chunk.score:.4f})")
            print(f"      {chunk.text[:300]}...")

    print("=" * 70)


print("display_response() defined.")

## Cell 7 — Example queries

Three example queries relevant to a banking regulatory context.
Edit `QUESTION` to test your own queries.

These queries are designed to exercise different parts of the regulatory corpus and demonstrate source attribution.

In [ ]:
# Query 1 — Capital adequacy
QUESTION = "What are the minimum capital adequacy requirements for registered banks?"

print(f"Querying: {QUESTION}")
print("Please wait...\n")

response = query(QUESTION)
display_response(response)

In [ ]:
# Query 2 — AML/CFT obligations
QUESTION = "What customer due diligence obligations apply under AML/CFT requirements?"

print(f"Querying: {QUESTION}")
print("Please wait...\n")

response = query(QUESTION)
display_response(response)

In [ ]:
# Query 3 — Operational risk
QUESTION = "What are the key requirements for operational risk management in banks?"

print(f"Querying: {QUESTION}")
print("Please wait...\n")

response = query(QUESTION)
display_response(response)

## Cell 8 — Interactive query

Enter your own question here. Useful for exploring the corpus interactively during a demo.

In [ ]:
# Edit this question and re-run the cell
YOUR_QUESTION = "What reporting obligations do banks have under prudential standards?"

response = query(YOUR_QUESTION)
display_response(response, show_chunks=True)

## Cell 9 — Retrieval diagnostic

Shows retrieved chunks without generating an answer. Useful for understanding what the vector store is returning and tuning TOP_K or chunk size.

In [ ]:
DIAGNOSTIC_QUERY = "liquidity risk management requirements"

print(f"Retrieval diagnostic for: '{DIAGNOSTIC_QUERY}'")
print(f"TOP_K = {TOP_K}")
print("-" * 60)

chunks = retrieve(DIAGNOSTIC_QUERY)

for i, chunk in enumerate(chunks, start=1):
    print(f"\n[{i}] Source : {chunk.source}, page {chunk.page}")
    print(f"    Score  : {chunk.score:.4f}")
    print(f"    Text   : {chunk.text[:400]}")